In [ ]:
import pandas as pd
from pathlib import Path

In [ ]:
# Caminhos do projeto (relativos à pasta notebooks/)
bronze_dir = Path("../dados/bronze")
silver_dir = Path("../dados/prata")
source_file = bronze_dir / "Controle de alertas - CMF-SEAMA.xlsx"

In [ ]:
# Lê a aba de respostas do Idaf
df_raw = pd.read_excel(source_file, sheet_name="RESPOSTA")

# Colunas do funil, com nomes padronizados (sem textos livres nem nomes)
response_columns = {
    "ID alerta": "id_alerta",
    "Enviado para fiscalização?": "enviado_fiscalizacao",
    "Motivo": "motivo",
    "Orgão responsável": "orgao_externo",
    "Data da fisc": "data_fiscalizacao",
    "Houve infração?": "houve_infracao",
    "nº da fisc. sem infração": "fiscalizacao_sem_infracao",
    "Fiscalizacao_IUF1": "fiscalizacao_com_infracao",
    "Tipo de infrator": "tipo_infrator",
}
df_responses = df_raw[list(response_columns)].rename(columns=response_columns)
df_responses["linha_excel"] = df_responses.index + 2  # cabeçalho + base 0

# Remove linhas em branco (sem ID e sem nenhuma informação de triagem)
check_blank = ["id_alerta", "enviado_fiscalizacao", "motivo", "houve_infracao"]
df_responses = df_responses.dropna(subset=check_blank, how="all")
print(len(df_responses), "linhas com conteúdo")

In [ ]:
# Padroniza as categorias: sem espaços nas pontas e em minúsculas
valid_values = {
    "enviado_fiscalizacao": {"sim", "não"},
    "motivo": {"falso positivo", "alerta tardio", "iuf tardio", "terra indígena"},
    "houve_infracao": {"sim", "não"},
    "tipo_infrator": {"pessoa física", "pessoa jurídica"},
    "orgao_externo": {"ibama"},
}
for column in valid_values:
    df_responses[column] = df_responses[column].astype("string").str.strip().str.lower()

df_responses["tipo_infrator"] = df_responses["tipo_infrator"].replace(
    {"pessoa fisica": "pessoa física"})

In [ ]:
# Marca cada linha inválida com o motivo
df_responses["motivo_quarentena"] = ""

no_id = df_responses["id_alerta"].isna()
df_responses.loc[no_id, "motivo_quarentena"] = "sem ID de alerta; "

for column, valid_set in valid_values.items():
    invalid = df_responses[column].notna() & ~df_responses[column].isin(valid_set)
    df_responses.loc[invalid, "motivo_quarentena"] = (
        df_responses.loc[invalid, "motivo_quarentena"] + f"{column} inválido; ")

# Separa a quarentena do restante
is_quarantine = df_responses["motivo_quarentena"] != ""
df_quarantine = df_responses[is_quarantine].copy()
df_responses = df_responses[~is_quarantine].drop(columns="motivo_quarentena")

print(len(df_responses), "linhas válidas |", len(df_quarantine), "em quarentena")
print(df_quarantine[["linha_excel", "motivo_quarentena"]].to_string())

In [ ]:
# Tipos
df_responses["id_alerta"] = df_responses["id_alerta"].astype("int64")

raw_dates = df_responses["data_fiscalizacao"]
df_responses["data_fiscalizacao"] = pd.to_datetime(raw_dates, errors="coerce")
lost_dates = raw_dates.notna() & df_responses["data_fiscalizacao"].isna()
print(lost_dates.sum(), "datas de fiscalização que não converteram")

In [ ]:
# Chave da fiscalização: o número com infração ou, se não houver, o sem infração
with_infraction = pd.to_numeric(df_responses["fiscalizacao_com_infracao"], errors="coerce")
without_infraction = pd.to_numeric(df_responses["fiscalizacao_sem_infracao"], errors="coerce")
df_responses["id_fiscalizacao"] = with_infraction.fillna(without_infraction).astype("Int64")
df_responses = df_responses.drop(columns=["fiscalizacao_com_infracao", "fiscalizacao_sem_infracao"])

In [ ]:
# Checagens bloqueantes: se falharem, o notebook para
df_alerts = pd.read_parquet(silver_dir / "alertas.parquet")

assert df_responses["id_alerta"].isin(df_alerts["id_alerta"]).all(), \
    "Há resposta para alerta que não existe na CONTROLE"

inspected = df_responses["houve_infracao"].notna()
assert df_responses.loc[inspected, "enviado_fiscalizacao"].eq("sim").fillna(False).all(), \
    "Há fiscalização registrada em alerta não enviado a campo"

# Checagens informativas: só reportam, não bloqueiam
print("Fiscalizações registradas:", inspected.sum())
print("  sem número de fiscalização:", df_responses.loc[inspected, "id_fiscalizacao"].isna().sum())
print("  sem data de fiscalização:", df_responses.loc[inspected, "data_fiscalizacao"].isna().sum())

In [ ]:
# Salva respostas válidas e quarentena na camada prata
df_responses.to_parquet(silver_dir / "respostas.parquet", index=False)
df_quarantine.astype("string").to_parquet(silver_dir / "quarentena_respostas.parquet", index=False)

print(len(df_responses), "respostas salvas")
df_responses.info()